## Heatmap of group differences (iNPH vs controls)
Edit **Cell 1 only** for day-to-day use. Cells 2-5 define the functions; Cell 6 runs everything.

**Expected file format** (one CSV per metric): one row per region, with at least a region name, a raw p-value and an effect size. Other columns (U, n, CI...) are allowed and are carried through untouched.

In [ ]:
# CELL 1 - imports and settings you will edit most often
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

# ---- 1. Your input files: one file per metric ----------------
# Key   = the label shown on the heatmap column (keep it short)
# Value = path to that metric's results file
# Order here = left-to-right order on the plot.
# (Edit the paths; add or remove metrics freely.)
FILES = {
    "FA": "DTI_fa",
    "MD": "DTI_md",
    "RD": "DTI_rd",
    "AD": "DTI_ad",
    # "VOL": "DTI_vol",
}

# ---- 2. Optional metric groups (not used with DTI only) ------
# Leave as None: no separators or group titles on the plot.
# Only fill it in if you later want grouped columns, e.g. {"FA": "DTI", "QA": "QSDR"}
METRIC_GROUPS = None

# ---- 3. What the columns in your files are called ------------
# Left side = name used inside this code. Right side = name in your CSV.
# If you rename a column in your files later, change it only here.
COLUMN_MAP = {
    "region": "Regions",
    "p":      "p",       # raw p-value
    "effect": "r",       # effect size (rank-biserial, + = higher in iNPH)
}

# ---- 4. Multiple-comparison correction -----------------------
FDR_SCOPE = "metric"   # "metric" = each metric its own family (your current plan)
                       # "all"    = one family for everything (strictest)
                       # ("group" also exists, but needs METRIC_GROUPS)
Q_THRESHOLD = 0.05

### Step 1 - read the files
One function reads a single file and renames its columns to standard names; the second stacks all metrics into one long table.

In [ ]:
# CELL 2 - STEP 1: read the files
def load_metric_file(path, label, column_map=COLUMN_MAP):
    """Read ONE results file and return a tidy table with standard column names.

    Any extra columns in the file (e.g. n_patients, n_controls, CI limits)
    are kept as-is, so you can add them to your files later without
    touching this function.
    """
    df = pd.read_csv(path)

    missing = [csv_name for csv_name in column_map.values() if csv_name not in df.columns]
    if missing:
        raise KeyError(
            f"{path}: missing column(s) {missing}. "
            f"Columns found: {list(df.columns)}. Fix COLUMN_MAP in Cell 1."
        )

    # rename CSV names -> standard names (region / p / effect)
    df = df.rename(columns={csv: std for std, csv in column_map.items()})
    df.insert(0, "metric", label)
    return df


def load_all_metrics(files=FILES):
    """Read every file in FILES and stack them into one long table."""
    tables = [load_metric_file(path, label) for label, path in files.items()]
    return pd.concat(tables, ignore_index=True)

### Step 2 - FDR correction
Benjamini-Hochberg. `FDR_SCOPE` in Cell 1 chooses which tests form one family.

In [ ]:
# CELL 3 - STEP 2: multiple-comparison correction (Benjamini-Hochberg)
def benjamini_hochberg(p):
    """Return BH-adjusted p-values (q-values). NaNs are ignored."""
    p = np.asarray(p, dtype=float)
    q = np.full_like(p, np.nan)
    ok = ~np.isnan(p)
    pv = p[ok]
    n = len(pv)
    order = np.argsort(pv)
    scaled = pv[order] * n / np.arange(1, n + 1)              # p * n / rank
    scaled = np.minimum.accumulate(scaled[::-1])[::-1]        # enforce monotonic
    adj = np.empty(n)
    adj[order] = np.minimum(scaled, 1.0)
    q[ok] = adj
    return q


def add_fdr(df, scope=FDR_SCOPE, metric_groups=METRIC_GROUPS):
    """Add a 'q' column. 'scope' decides which tests are corrected together."""
    df = df.copy()
    if scope == "metric":
        df["family"] = df["metric"]
    elif scope == "group":
        if not metric_groups:
            raise ValueError("FDR_SCOPE='group' needs METRIC_GROUPS to be set.")
        df["family"] = df["metric"].map(metric_groups)
    elif scope == "all":
        df["family"] = "all"
    else:
        raise ValueError("scope must be 'metric', 'group' or 'all'")

    df["q"] = df.groupby("family")["p"].transform(lambda s: benjamini_hochberg(s.values))
    return df

### Step 3 - panels and row labels
Turns region names into a panel (Corpus callosum / Left / Right) and a row label. Edit `default_panel_fn` if your naming changes.

In [ ]:
# CELL 4 - STEP 3: decide which panel / row each region goes to
def default_panel_fn(region):
    """Return (panel_name, row_label) for one region name.

    EDIT THIS if your naming changes. Current rules:
      CC_anterior -> panel 'Corpus callosum', row 'anterior'
      L_3a        -> panel 'Left',            row '3a'
      R_3a        -> panel 'Right',           row '3a'
      anything else -> panel 'Other', row = full name
    Left and Right panels share row labels, so 3a sits at the same
    height in both and you can compare hemispheres by eye.
    """
    m = re.match(r"^CC_(.+)$", region)
    if m:
        return "Corpus callosum", m.group(1)
    m = re.match(r"^([LR])_(.+)$", region)
    if m:
        return ("Left" if m.group(1) == "L" else "Right"), m.group(2)
    return "Other", region


def add_panels(df, panel_fn=default_panel_fn):
    df = df.copy()
    out = df["region"].apply(panel_fn)
    df["panel"] = [o[0] for o in out]
    df["row"] = [o[1] for o in out]
    return df

### Step 4 - drawing
`draw_panel` draws one block. `plot_heatmap` arranges the blocks and the legend. Its keyword arguments (colour range, sorting, hide non-significant rows, cell size, title, saving) are the customization knobs.

In [ ]:
# CELL 5 - STEP 4: drawing (small helpers first, main function last)
def stars_for(q, thresholds=((0.001, "***"), (0.01, "**"), (0.05, "*"))):
    """Turn a q-value into '', '*', '**' or '***'. Only q below Q_THRESHOLD counts."""
    if np.isnan(q) or q >= Q_THRESHOLD:
        return ""
    for cutoff, s in thresholds:
        if q < cutoff:
            return s
    return "*"


def make_matrices(df, panel, rows, metrics):
    """Build the effect-size and q matrices (rows x metrics) for one panel."""
    sub = df[df["panel"] == panel]
    eff = sub.pivot(index="row", columns="metric", values="effect").reindex(index=rows, columns=metrics)
    q = sub.pivot(index="row", columns="metric", values="q").reindex(index=rows, columns=metrics)
    return eff, q


def draw_panel(ax, eff, q, *, cmap, vlim, show_row_labels, col_groups, annot_size):
    """Draw ONE heatmap block (colored cells + stars + labels) on 'ax'."""
    n_rows, n_cols = eff.shape
    im = ax.imshow(eff.values.astype(float), cmap=cmap, vmin=-vlim, vmax=vlim,
                   aspect="auto", interpolation="nearest")

    # thin white grid between cells
    ax.set_xticks(np.arange(-.5, n_cols, 1), minor=True)
    ax.set_yticks(np.arange(-.5, n_rows, 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=0.5)
    ax.tick_params(which="both", length=0)

    # axis labels
    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(eff.columns, rotation=90, fontsize=8)
    ax.xaxis.tick_top()
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(eff.index if show_row_labels else [], fontsize=6)
    for side in ax.spines.values():
        side.set_visible(False)

    # stars on significant cells (white text on dark cells so it stays readable)
    for i in range(n_rows):
        for j in range(n_cols):
            s = stars_for(q.iat[i, j])
            if s:
                dark = abs(eff.iat[i, j]) > 0.6 * vlim
                ax.text(j, i, s, ha="center", va="center", fontsize=annot_size,
                        color="white" if dark else "black", fontweight="bold")

    # group separators + group titles (e.g. DTI | QSDR)
    if col_groups:
        names = [col_groups.get(m, "") for m in eff.columns]
        start = 0
        for k in range(1, n_cols + 1):
            if k == n_cols or names[k] != names[start]:
                if k < n_cols:
                    ax.axvline(k - 0.5, color="black", linewidth=1.2)
                longest = max(len(str(c)) for c in eff.columns)
                ax.annotate(names[start], xy=((start + k - 1) / 2, 1.0),
                            xycoords=("data", "axes fraction"),
                            xytext=(0, 8 + longest * 5), textcoords="offset points",
                            ha="center", va="bottom", fontsize=9, fontweight="bold")
                start = k
    return im


def plot_heatmap(results, *,
                 vlim=1.0,                 # color scale runs from -vlim to +vlim
                 cmap="RdBu_r",            # red = higher in iNPH, blue = lower
                 only_significant=False,   # True = hide rows with no star anywhere
                 sort_by=None,             # e.g. "QA" = order rows by that metric's effect
                 top_panels=("Corpus callosum", "Other"),   # drawn as a strip on top
                 col_groups=METRIC_GROUPS,
                 cell_w=0.42, cell_h=0.16,  # inches per cell; shrink cell_h for a shorter figure
                 annot_size=6,
                 effect_label="Rank-biserial r  (+ = higher in iNPH)",
                 title=None,
                 save_as=None):            # e.g. "heatmap.png" or ["heatmap.png", "heatmap.pdf"]
    """Draw the heatmap. 'results' must come from add_panels(add_fdr(load_all_metrics()))."""
    df = results.copy()
    metrics = list(dict.fromkeys(df["metric"]))              # keeps FILES order

    # ---- which rows to show, and in what order -------------
    rows = list(dict.fromkeys(df["row"]))                    # order of first appearance
    if only_significant:
        keep = set(df.loc[df["q"] < Q_THRESHOLD, "row"])
        rows = [r for r in rows if r in keep]
    if sort_by:
        mean_eff = (df[df["metric"] == sort_by].groupby("row")["effect"].mean())
        rows = sorted(rows, key=lambda r: mean_eff.get(r, np.nan), reverse=True)
    df = df[df["row"].isin(rows)]

    # ---- split panels into a top strip and a bottom block ---
    panels = list(dict.fromkeys(df["panel"]))
    top = [p for p in panels if p in top_panels]
    bottom = [p for p in panels if p not in top_panels]
    rows_of = {p: [r for r in rows if r in set(df.loc[df["panel"] == p, "row"])] for p in panels}
    n_top = max([len(rows_of[p]) for p in top], default=0)
    n_bot = max([len(rows_of[p]) for p in bottom], default=0)

    # ---- figure layout -------------------------------------
    # Every panel gets 'pad' empty rows above it (room for metric labels),
    # then exactly one grid row per heatmap row, so all cells are the same size.
    pad = 6
    n_cols_grid = max(len(bottom), 2)
    fig_w = 1.6 + n_cols_grid * (len(metrics) * cell_w + 0.6)
    fig_h = 1.2 + (2 * pad + n_top + n_bot + 2) * cell_h
    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = GridSpec(2, n_cols_grid, figure=fig,
                  height_ratios=[pad + max(n_top, 6), pad + n_bot],
                  hspace=0.02, wspace=0.12, top=0.97, bottom=0.03)

    def panel_axes(grid_cell, n_rows):
        """Axes with exactly n_rows grid rows, leaving 'pad' blank rows above."""
        sub = grid_cell.subgridspec(2, 1, height_ratios=[pad, n_rows], hspace=0)
        return fig.add_subplot(sub[1, 0])

    im = None
    # top strip (e.g. corpus callosum), left column
    for p in top[:1]:
        eff, q = make_matrices(df, p, rows_of[p], metrics)
        ax = panel_axes(gs[0, 0], len(rows_of[p]))
        im = draw_panel(ax, eff, q, cmap=cmap, vlim=vlim, show_row_labels=True,
                        col_groups=col_groups, annot_size=annot_size)
        ax.set_xlabel(p, fontsize=9, fontweight="bold")

    # bottom block: one heatmap per panel (e.g. Left | Right)
    for k, p in enumerate(bottom):
        eff, q = make_matrices(df, p, rows_of[p], metrics)
        ax = panel_axes(gs[1, k], len(rows_of[p]))
        im = draw_panel(ax, eff, q, cmap=cmap, vlim=vlim, show_row_labels=(k == 0),
                        col_groups=col_groups, annot_size=annot_size)
        ax.set_xlabel(p, fontsize=9, fontweight="bold")

    # legend cell: colorbar + significance key
    lax = fig.add_subplot(gs[0, -1]); lax.axis("off")
    cax = lax.inset_axes([0.15, 0.55, 0.7, 0.10])
    cb = fig.colorbar(im, cax=cax, orientation="horizontal")
    cb.set_label(effect_label, fontsize=8); cb.ax.tick_params(labelsize=7)
    scope_text = {"metric": "within each metric", "group": "within each metric group",
                  "all": "across all tests"}[FDR_SCOPE]
    lax.text(0.15, 0.25, f"*  q < 0.05    **  q < 0.01    ***  q < 0.001\n"
                         f"Benjamini-Hochberg FDR, {scope_text}\n"
                         f"Grey = test not run / tractography failed",
             fontsize=7, va="top", transform=lax.transAxes)

    if title:
        fig.suptitle(title, fontsize=11, fontweight="bold", y=0.995)
    if save_as:
        for f in ([save_as] if isinstance(save_as, str) else save_as):
            fig.savefig(f, dpi=300, bbox_inches="tight")
    return fig

### Run it

In [ ]:
# CELL 6 - RUN IT
results = add_panels(add_fdr(load_all_metrics()))

fig = plot_heatmap(results, only_significant=False, title=None, save_as="heatmap.png")
plt.show()

# Full table for the supplementary material (raw p, q, effect size, plus any extra columns):
results.to_csv("results_with_fdr.csv", index=False)